# 1.5b Reddit — Sentiment Analysis (RoBERTa Twitter)

**Temporary exploratory notebook** — applies `cardiffnlp/twitter-roberta-base-sentiment-latest`
to Reddit posts + comments and reproduces the same graphs as `1.5_sentiment_analysis.ipynb`.

| Field | Details |
|---|---|
| **Model** | `cardiffnlp/twitter-roberta-base-sentiment-latest` |
| **Labels** | Negative · Neutral · Positive |
| **Compound** | `score_positive − score_negative` ∈ [−1, +1] |
| **Input** | `Data/2_Silver/Reddit/reddit_posts_clean.parquet` + `reddit_comments_clean.parquet` |
| **Batch size** | 64 (GPU auto-detected) |

> **Note:** For 1.4 M rows this takes ~30–60 min on CPU. Set `SAMPLE_N` below to a smaller
> number (e.g. `50_000`) for a quick test run.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

sys.path.insert(0, os.path.abspath('../..'))
from house_style import (
    apply_style, REPUBLICAN, DEMOCRAT, NEUTRAL, ACCENT,
    TEXT_PRIMARY, TEXT_MUTED, BG_DARK, BG_PANEL,
    EVENTS, add_events,
)
apply_style()

# ── Paths ──────────────────────────────────────────────────────────────────────
POSTS_PATH    = '../../Data/2_Silver/Reddit/reddit_posts_clean.parquet'
COMMENTS_PATH = '../../Data/2_Silver/Reddit/reddit_comments_clean.parquet'
VADER_CSV     = '../../Data/2_Silver/Reddit/sentiment_reddit.csv'

# Set to None to process all rows; set to e.g. 50_000 for a quick test
SAMPLE_N   = None
BATCH_SIZE = 64
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Colour helpers ─────────────────────────────────────────────────────────────
BUZZ_COLORS = {'TrumpBuzz': REPUBLICAN, 'HarrisBuzz': DEMOCRAT, 'ElectionBuzz': NEUTRAL}
BUZZ_ORDER  = ['TrumpBuzz', 'HarrisBuzz', 'ElectionBuzz']
POL_COLORS  = {'Positive': '#2ecc71', 'Neutral': TEXT_MUTED, 'Negative': REPUBLICAN}
PURPLE      = '#9b5de5'

def event_legend(fig, ax, loc='upper left'):
    data_h = ax.get_legend_handles_labels()
    ev_h   = [mlines.Line2D([], [], color=c, linestyle=':', linewidth=1.8, label=lbl)
              for lbl, _, c in EVENTS]
    ax.legend(handles=data_h[0], labels=data_h[1], facecolor=BG_DARK,
              labelcolor=TEXT_PRIMARY, loc=loc)
    fig.legend(handles=ev_h, loc='lower center', bbox_to_anchor=(0.5, 0.0),
               ncol=min(len(EVENTS), 4), facecolor=BG_PANEL, edgecolor=TEXT_MUTED,
               labelcolor=TEXT_PRIMARY, fontsize=8.5, framealpha=0.95, borderpad=0.8)

print(f'Device : {DEVICE}')
print('Setup complete.')

## 1. Load data

In [ ]:
posts    = pd.read_parquet(POSTS_PATH)
comments = pd.read_parquet(COMMENTS_PATH)

posts['source']    = 'post'
comments['source'] = 'comment'

keep = ['id', 'author', 'created_utc', 'subreddit', 'score',
        'text', 'text_clean', 'word_count', 'candidate', 'source']

df = pd.concat([posts[keep], comments[keep]], ignore_index=True)
df['date']       = pd.to_datetime(df['created_utc'], utc=True).dt.normalize()
df['text_clean'] = df['text_clean'].fillna('')

df = df[(df['date'] >= '2024-07-05') & (df['date'] <= '2024-11-04')].reset_index(drop=True)

if SAMPLE_N is not None:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
    print(f'>>> SAMPLE mode: using {len(df):,} rows <<<')

print(f'Total rows : {len(df):,}  ({len(posts):,} posts + {len(comments):,} comments)')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print(df['candidate'].value_counts())

## 2. Load RoBERTa model

In [ ]:
print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(DEVICE)
model.eval()

LABELS = [model.config.id2label[i] for i in range(model.config.num_labels)]
print(f'Labels : {LABELS}')
print('Model ready.')

## 3. Run inference (batched)

In [ ]:
def roberta_batch(texts, batch_size=BATCH_SIZE):
    """Score texts in batches; returns list of {label: prob} dicts."""
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Scoring'):
        batch = list(texts[i : i + batch_size])
        enc   = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            max_length=128,
            padding=True,
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = softmax(logits.cpu().numpy(), axis=1)
        for row in probs:
            results.append({lbl: float(p) for lbl, p in zip(LABELS, row)})
    return results

scores    = roberta_batch(df['text_clean'].tolist())
scores_df = pd.DataFrame(scores)
df        = pd.concat([df, scores_df], axis=1)

# Normalise label names → Negative / Neutral / Positive
label_map = {}
for lbl in LABELS:
    l = lbl.lower()
    if 'neg' in l:   label_map[lbl] = 'Negative'
    elif 'neu' in l: label_map[lbl] = 'Neutral'
    else:            label_map[lbl] = 'Positive'
df.rename(columns=label_map, inplace=True)

df['roberta_compound'] = df['Positive'] - df['Negative']
df['roberta_label']    = df[['Negative', 'Neutral', 'Positive']].idxmax(axis=1)

print('RoBERTa distribution:')
print(df['roberta_label'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nMean compound: {df["roberta_compound"].mean():+.4f}')

## 4. Distribution of compound scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for cand, color in BUZZ_COLORS.items():
    sub = df[df['candidate'] == cand]['roberta_compound']
    ax.hist(sub, bins=40, color=color, alpha=0.5, label=cand, edgecolor='none')
ax.axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.6)
ax.set_title('RoBERTa Twitter — Distribution of Compound Scores',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_xlabel('Compound score (Positive − Negative)', color=TEXT_MUTED)
ax.set_ylabel('Count', color=TEXT_MUTED)
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()

## 5. Polarity share per buzz group

In [ ]:
label_shares = (
    df.groupby(['candidate', 'roberta_label']).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .reindex(BUZZ_ORDER)
)
for col in ['Positive', 'Neutral', 'Negative']:
    if col not in label_shares.columns:
        label_shares[col] = 0.0
label_shares = label_shares[['Positive', 'Neutral', 'Negative']]

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
x, bottom = np.arange(len(BUZZ_ORDER)), np.zeros(len(BUZZ_ORDER))
for label, color in POL_COLORS.items():
    vals = label_shares[label].values
    ax.bar(x, vals, bottom=bottom, color=color, label=label, edgecolor='none')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(x[i], b + v/2, f'{v:.0%}', ha='center', va='center',
                    color='white', fontsize=9)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(BUZZ_ORDER, color=TEXT_MUTED)
ax.set_ylabel('Share', color=TEXT_MUTED)
ax.set_title('RoBERTa Twitter — Polarity Share per Buzz Group',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.tick_params(colors=TEXT_MUTED)
ax.legend(facecolor=BG_DARK, labelcolor=TEXT_PRIMARY)
plt.tight_layout(); plt.show()

## 6. Daily compound score over time (7d rolling avg)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for group, color in BUZZ_COLORS.items():
    daily = df[df['candidate'] == group].groupby('date')['roberta_compound'].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=group)
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
add_events(ax)
ax.set_title('RoBERTa Twitter — Daily Compound Score per Buzz Group (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Compound (Pos − Neg)', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()

## 7. Per-class probability over time (7d rolling avg)

In [ ]:
CLASS_COLORS = {'Positive': '#2ecc71', 'Neutral': TEXT_MUTED, 'Negative': REPUBLICAN}
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
for cls, color in CLASS_COLORS.items():
    daily = df.groupby('date')[cls].mean()
    roll  = daily.rolling(7, min_periods=1).mean()
    ax.plot(daily.index, daily.values, color=color, alpha=0.15, linewidth=1)
    ax.plot(roll.index, roll.values, color=color, linewidth=2, label=cls)
add_events(ax)
ax.set_title('RoBERTa Twitter — Class Probabilities over Time (7d rolling avg)',
             color=TEXT_PRIMARY, fontsize=12, fontweight='bold')
ax.set_ylabel('Mean probability', color=TEXT_MUTED)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.tick_params(axis='x', rotation=30, colors=TEXT_MUTED)
ax.tick_params(axis='y', colors=TEXT_MUTED)
event_legend(fig, ax)
plt.tight_layout(rect=[0, 0.18, 1, 1]); plt.show()

## 9. Save (temporary)

In [ ]:
OUT = '../../Data/2_Silver/Reddit/sentiment_reddit_roberta.csv'
save_cols = ['date', 'candidate', 'subreddit', 'source',
             'Negative', 'Neutral', 'Positive',
             'roberta_compound', 'roberta_label']
df[save_cols].to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows → {OUT}')